# Multi-Head Cross Attention

Cross-attention dùng trong decoder: Query lấy từ decoder (`y`), còn Key/Value lấy từ output của encoder (`x`). Nhờ đó decoder có thể "tham chiếu" tới toàn bộ chuỗi input khi sinh output.

In [1]:
import numpy as np

Các hàm/lớp dùng lại y hệt notebook Self Attention / Multi-head Attention: `softmax`, `scaled_dot_product_attention`, `Parameter`, `Linear`.

In [2]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.shape[-1]
    scaled = np.matmul(q, np.swapaxes(k, -2, -1)) / np.sqrt(d_k)
    if mask is not None:
        scaled = scaled + mask
    attention = softmax(scaled, axis=-1)
    out = np.matmul(attention, v)
    return out, attention


class Parameter:
    def __init__(self, data):
        self.data = data
        self.grad = None


class Linear:
    def __init__(self, in_features, out_features):
        self.W = Parameter(np.random.randn(in_features, out_features) / np.sqrt(in_features))
        self.b = Parameter(np.zeros((out_features,)))

    def __call__(self, x):
        return np.matmul(x, self.W.data) + self.b.data

`MultiHeadCrossAttention`: chỉ có 1 layer `q_layer` chiếu `y` (decoder) thành Q, và 1 layer `kv_layer` chiếu `x` (encoder output) thành K, V gộp chung — khác self-attention (chỉ 1 input nên dùng chung 1 layer `qkv_linear`).

In [ ]:
class MultiHeadCrossAttention():

    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.kv_layer = Linear(d_model, 2 * d_model) 
        self.q_layer = Linear(d_model, d_model)
        self.linear_layer = Linear(d_model, d_model)

    def forward(self, x, y, mask=None):
        batch_size, sequence_length, d_model = x.shape 
        print(f"x.shape: {x.shape}")
        kv = self.kv_layer(x) 
        print(f"kv.shape: {kv.shape}")
        q = self.q_layer(y) 
        print(f"q.shape: {q.shape}")
        kv = kv.reshape(batch_size, sequence_length, self.num_heads, 2 * self.head_dim)  
        q = q.reshape(batch_size, sequence_length, self.num_heads, self.head_dim) 
        kv = np.transpose(kv, (0, 2, 1, 3)) 
        q = np.transpose(q, (0, 2, 1, 3)) 
        k, v = np.split(kv, 2, axis=-1) 
        values, attention = scaled_dot_product_attention(q, k, v, mask) 
        print(f"values.shape: {values.shape}, attention.shape: {attention.shape}")
        values = values.reshape(batch_size, sequence_length, d_model) 
        out = self.linear_layer(values)  
        print(f"out.shape after passing through linear layer: {out.shape}")
        return out  

### Input

`x` đóng vai trò output của encoder, `y` là input hiện tại của decoder — cùng shape `(batch_size, sequence_length, d_model)`.

In [4]:
d_model = 512
num_heads = 8
batch_size = 30
max_sequence_length = 200

x = np.random.randn(batch_size, max_sequence_length, d_model)  # encoder output
y = np.random.randn(batch_size, max_sequence_length, d_model)  # decoder input
model = MultiHeadCrossAttention(d_model, num_heads)
out = model.forward(x, y)
out.shape, type(out)

x.shape: (30, 200, 512)
kv.shape: (30, 200, 1024)
q.shape: (30, 200, 512)
values.shape: (30, 8, 200, 64), attention.shape: (30, 8, 200, 200)
out.shape after passing through linear layer: (30, 200, 512)


((30, 200, 512), <class 'numpy.ndarray'>)